In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
import aotools
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator, FuncFormatter
import VisAnalysis as Vis
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.interpolate import interp1d
%matplotlib qt

In [2]:
NL = Vis.load_measurements(base_dir="D:\PhD Thesis Writing\Data", date_str="2025-11-28", subfolder="Night2Long", verbose=True)
NLF = Vis.load_measurements(base_dir="D:\PhD Thesis Writing\Data", date_str="2025-11-28", subfolder="Night2LongFast", verbose=True)
NLVF = Vis.load_measurements(base_dir="D:\PhD Thesis Writing\Data", date_str="2025-11-28", subfolder="Night2LongVeryFast", verbose=True)

Loaded 13 measurements from D:\PhD Thesis Writing\Data\2025-11-28\Night2Long
Loaded 13 measurements from D:\PhD Thesis Writing\Data\2025-11-28\Night2LongFast
Loaded 13 measurements from D:\PhD Thesis Writing\Data\2025-11-28\Night2LongVeryFast


In [4]:
DF = Vis.load_measurements(base_dir="D:\PhD Thesis Writing\Data", date_str="2025-11-26", subfolder="Day2Long", verbose=True)

Loaded 13 measurements from D:\PhD Thesis Writing\Data\2025-11-26\Day2Long


In [6]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter

def frac_to_percent(x, pos):
    return f"{int(round(100 * x))}"
# ----------------------------
# C5 NIGHT SLOW VS FAST - EM
# ----------------------------

d1 = np.load('NLVF_10_em_Numba.npy', allow_pickle=True).tolist()
d2 = np.load('NL_10_em_Numba.npy', allow_pickle=True).tolist()
x1 = d1[0][2][0]
y1 = d1[0][2][1]
x2 = d2[0][2][0]
y2 = d2[0][2][1]

DB_1 = Vis.convert_to_diff_db(np.load('rx_300m_night_2_long_veryfast_etas.npy'))
ts_1 = [i / 1000 for i in range(len(DB_1))]

DB_2 = Vis.convert_to_diff_db(np.load('rx_300m_night_2_long_etas.npy'))
ts_2 = [i / 1000 for i in range(len(DB_2))]

# ----------------------------
# Matplotlib style
# ----------------------------
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=18)
plt.rc('axes', labelsize=16)

fig = plt.figure(figsize=(8, 6))

# Slightly wider histogram column, slightly more vertical spacing
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[3.5, 1],
    width_ratios=[6.2, 1.4],
    hspace=0.02,
    wspace=0.06
)

ax_main      = fig.add_subplot(gs[0, 0])                 # top-left
ax_sub       = fig.add_subplot(gs[1, 0], sharex=ax_main) # bottom-left
ax_hist_main = fig.add_subplot(gs[0, 1], sharey=ax_main) # top-right (QBER hist)
ax_hist_sub  = fig.add_subplot(gs[1, 1], sharey=ax_sub)  # bottom-right (Δη hist)

# ----------------------------
# Plot data
# ----------------------------
line1, = ax_main.plot(x1, y1 * 100, label='V. Fast')
line2, = ax_main.plot(x2, y2 * 100, label='Slow')

ax_main.set_ylim(0, 50)
ax_main.set_xlim(0, 200)

ax_sub.plot(ts_1, DB_1, color=line1.get_color())
ax_sub.plot(ts_2,  DB_2,  color=line2.get_color())

# ----------------------------
# Labels / title
# ----------------------------
ax_sub.set_xlabel('Time [$s$]')
ax_main.set_ylabel('QBER [$\\%$]')
ax_sub.set_ylabel('$\\Delta\\eta$ [db]')
ax_main.set_title('QBER/time - 10Hz, Night Slow/Very Fast')

# Hide top x tick labels (shared x)
plt.setp(ax_main.get_xticklabels(), visible=False)
ax_main.tick_params(axis='x', which='both', length=0)

# ----------------------------
# Histogram formatting (reduce clutter)
# ----------------------------
for axh in (ax_hist_main, ax_hist_sub):
    axh.grid(False)
    axh.tick_params(axis='y', labelleft=False, left=False)
    axh.tick_params(axis='y', which='both', length=0)
    axh.minorticks_off()
    axh.set_xlabel("Fraction")

# Avoid tick/label collision between histogram panels:
ax_hist_main.tick_params(axis='x', which='both', labelbottom=False)
ax_hist_main.set_xlabel("")

# ----------------------------
# Helpers
# ----------------------------
def _finite(a):
    a = np.asarray(a)
    return a[np.isfinite(a)]

def _rgba(color, alpha):
    return mcolors.to_rgba(color, alpha=alpha)

def choose_bins_freedman_diaconis(data, data_min, data_max, min_bins=40, max_bins=180):
    """
    Freedman–Diaconis rule, clipped to a sensible range.
    Uses data spread (IQR) and sample size, NOT axis length.
    Falls back to sqrt(N) if IQR==0.
    """
    data = _finite(data)
    n = data.size
    if n < 2:
        return min_bins

    q25, q75 = np.percentile(data, [25, 75])
    iqr = q75 - q25
    data_range = float(data_max - data_min)

    if iqr <= 0 or data_range <= 0:
        nb = int(np.sqrt(n))
    else:
        bin_width = 2 * iqr * (n ** (-1/3))
        if bin_width <= 0:
            nb = int(np.sqrt(n))
        else:
            nb = int(np.ceil(data_range / bin_width))

    return int(np.clip(nb, min_bins, max_bins))

def fraction_hist_with_stats(
    axh, data, bins, base_color,
    fill_alpha=0.35,
    lw=1.2,
    sigma_band_alpha=0.12
):
    data = _finite(data)
    if data.size == 0:
        return 0.0, np.nan, np.nan

    mu = float(np.mean(data))
    sigma = float(np.std(data))

    # Std band (lowest)
    axh.axhspan(
        mu - sigma, mu + sigma,
        facecolor=base_color, alpha=sigma_band_alpha,
        edgecolor='none', zorder=0
    )

    # Fraction-per-bin normalization
    weights = np.ones_like(data, dtype=float) / data.size
    light = _rgba(base_color, fill_alpha)

    # Histogram fill + outline (above band)
    axh.hist(
        data,
        bins=bins,
        orientation='horizontal',
        weights=weights,
        density=False,
        histtype='stepfilled',
        facecolor=light,
        edgecolor=light,
        linewidth=lw,
        zorder=3
    )

    # Mean line (topmost so it can't be hidden by the other series)
    axh.axhline(mu, color=base_color, linewidth=2.8, zorder=4)

    h, _ = np.histogram(data, bins=bins, weights=weights, density=False)
    peak = float(h.max()) if h.size else 0.0
    return peak, mu, sigma


# ----------------------------
# Build bins and plot histograms
# ----------------------------
fig.canvas.draw()  # ensure limits are finalized

# Bins match each row's y-range (so the histogram aligns perfectly with the axis it belongs to)
ymin_m, ymax_m = ax_main.get_ylim()
ymin_s, ymax_s = ax_sub.get_ylim()

# Choose bin counts using a stable rule (based on sample size/spread)
nb_main = choose_bins_freedman_diaconis(np.r_[y1*100, y2*100], ymin_m, ymax_m)
nb_sub  = choose_bins_freedman_diaconis(np.r_[DB_1, DB_2], ymin_s, ymax_s)

bins_main = np.linspace(ymin_m, ymax_m, nb_main)
bins_sub  = np.linspace(ymin_s, ymax_s, nb_sub)

c0 = line1.get_color()
c1 = line2.get_color()

# Plot upper histogram (QBER) and capture stats for legend text
p1, mu1, s1 = fraction_hist_with_stats(ax_hist_main, y1 * 100, bins_main, base_color=c0)
p2, mu2, s2 = fraction_hist_with_stats(ax_hist_main, y2 * 100, bins_main, base_color=c1)

# Plot lower histogram (Δη)
p3, _, _ = fraction_hist_with_stats(ax_hist_sub, DB_1, bins_sub, base_color=c0)
p4, _, _ = fraction_hist_with_stats(ax_hist_sub, DB_2,  bins_sub, base_color=c1)

# Shared horizontal axis for BOTH histogram panels
peak = max(p1, p2, p3, p4)
xmax = 1.10 * peak if peak > 0 else 1.0
ax_hist_main.set_xlim(0, xmax)
ax_hist_sub.set_xlim(0, xmax)
ax_hist_main.xaxis.set_major_formatter(FuncFormatter(frac_to_percent))
ax_hist_sub.xaxis.set_major_formatter(FuncFormatter(frac_to_percent))

ax_hist_sub.set_xlabel('Fraction [%]')

# ----------------------------
# Legend on TOP of upper histogram with μ, σ for the upper plots
# ----------------------------
legend_labels = [
    f"V. Fast: μ={mu1:.2f}%, σ={s1:.2f}%",
    f"Slow: μ={mu2:.2f}%, σ={s2:.2f}%"
]
ax_hist_main.legend(
    [line1, line2],
    legend_labels,
    loc="lower center",
    bbox_to_anchor=(-1.45, 0.01),  # place just above the upper histogram
    fontsize=12,
    frameon=True,
    ncol=1,
    borderaxespad=0.0
)

# ----------------------------
# Margins
# ----------------------------
fig.subplots_adjust(left=0.12, right=0.95, bottom=0.10, top=0.92)

plt.show()
plt.savefig('C5_NightF_10Hz_SvsVF.pdf', dpi=300)


In [19]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
from matplotlib import transforms
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

def _darken(color, factor=0.65):
    """
    Darken an RGB color by multiplying its channels by factor (<1 darker).
    Keeps hue consistent. factor=0.65 is a good default.
    """
    r, g, b = mcolors.to_rgb(color)
    return (r * factor, g * factor, b * factor)

from matplotlib import transforms
from matplotlib.lines import Line2D

def draw_stats_ticks_at_main_rhs(fig, ax_main_col, ax_hist, mu, sigma, color, cfg):
    """
    Draw short horizontal tick(s) at y=mu (and optionally y=mu±sigma),
    with x anchored to the right edge of the MAIN plot column (ax_main_col),
    and shifted by cfg["x_offset"] in figure coordinates.
    """
    bbox_main = ax_main_col.get_position()
    x_anchor = bbox_main.x1
    x0 = x_anchor + cfg["x_offset"]
    half_len = 0.5 * cfg["tick_len"]

    trans = transforms.blended_transform_factory(fig.transFigure, ax_hist.transData)

    # Mean tick
    fig.add_artist(Line2D(
        [x0 - half_len, x0 + half_len],
        [mu, mu],
        transform=trans,
        color=color,
        alpha=cfg["mean_alpha"],
        linewidth=cfg["mean_lw"],
        solid_capstyle='butt',
        zorder=cfg.get("zorder", 20),
        clip_on=False
    ))

    # Optional ±σ ticks
    if cfg.get("show_sigma", False) and np.isfinite(sigma) and sigma > 0:
        for y in (mu - sigma, mu + sigma):
            fig.add_artist(Line2D(
                [x0 - half_len, x0 + half_len],
                [y, y],
                transform=trans,
                color=color,
                alpha=cfg["sigma_alpha"],
                linewidth=cfg["sigma_lw"],
                linestyle=cfg.get("sigma_ls", "-"),
                solid_capstyle='butt',
                zorder=cfg.get("zorder", 20),
                clip_on=False
            ))


def frac_to_percent(x, pos):
    return f"{int(round(100 * x))}"


def _finite(a):
    a = np.asarray(a)
    return a[np.isfinite(a)]


def _rgba(color, alpha):
    return mcolors.to_rgba(color, alpha=alpha)


def choose_bins_freedman_diaconis(data, data_min, data_max,
                                  min_bins=80, max_bins=260):
    data = _finite(data)
    n = data.size
    if n < 2:
        return min_bins

    q25, q75 = np.percentile(data, [25, 75])
    iqr = q75 - q25
    data_range = float(data_max - data_min)

    if iqr <= 0 or data_range <= 0:
        nb = int(np.sqrt(n))
    else:
        bin_width = 2 * iqr * (n ** (-1 / 3))
        nb = int(np.ceil(data_range / bin_width)) if bin_width > 0 else int(np.sqrt(n))

    return int(np.clip(nb, min_bins, max_bins))

def fraction_hist_with_stats(
    axh, data, bins, base_color,
    outline_alpha=0.55,   # outline of full distribution
    outline_lw=1.0,
    core_alpha=0.55,      # fill only within ±1σ
):
    data = _finite(data)
    if data.size == 0:
        return 0.0, np.nan, np.nan

    mu = float(np.mean(data))
    sigma = float(np.std(data))

    # Normalize to fraction per bin
    weights = np.ones_like(data, dtype=float) / data.size

    # --- Full distribution: OUTLINE ONLY ---
    outline_col = _rgba(base_color, outline_alpha)
    axh.hist(
        data,
        bins=bins,
        orientation='horizontal',
        weights=weights,
        histtype='step',
        color=outline_col,
        linewidth=outline_lw,
        zorder=2
    )

    # --- Core region: FILL ONLY within ±1σ ---
    if np.isfinite(sigma) and sigma > 0:
        mask = (data >= mu - sigma) & (data <= mu + sigma)
        if np.any(mask):
            core_col = _rgba(base_color, core_alpha)
            axh.hist(
                data[mask],
                bins=bins,
                orientation='horizontal',
                weights=weights[mask],
                histtype='stepfilled',
                facecolor=core_col,
                edgecolor='none',
                zorder=1
            )

    # peak for shared x-lims
    h, _ = np.histogram(data, bins=bins, weights=weights)
    peak = float(h.max()) if h.size else 0.0

    return peak, mu, sigma





def apply_legend(ax, handles, labels, legend_cfg):
    """
    legend_cfg examples:
      {"loc":"lower center", "bbox_to_anchor":(-1.45, 0.01), "fontsize":12, "frameon":True, "ncol":1}
      {"loc":"upper right"}
      {"enabled": False}
    """
    if legend_cfg is None:
        legend_cfg = {}
    if legend_cfg.get("enabled", True) is False:
        return None

    # defaults
    cfg = dict(fontsize=12, frameon=True)
    cfg.update(legend_cfg)

    # remove non-legend keys if present
    cfg.pop("enabled", None)

    return ax.legend(handles, labels, **cfg)


In [21]:
# ----------------------------
# Load data
# ----------------------------

d1 = np.load('NNF_10_em_Numba.npy', allow_pickle=True).tolist()
d2 = np.load('NNF_10_nm_Numba.npy', allow_pickle=True).tolist()

x1 = d1[0][2][0]
y1 = d1[0][2][1]
x2 = d2[0][2][0]
y2 = d2[0][2][1]

DB_1 = Vis.convert_to_diff_db(np.load('rx_300m_night_2_long_etas.npy'))
ts_1 = [i / 1000 for i in range(len(DB_1))]

DB_2 = Vis.convert_to_diff_db(np.load('rx_300m_night_2_long_etas.npy'))
ts_2 = [i / 1000 for i in range(len(DB_2))]

# ----------------------------
# Plot configuration (edit THIS cell only)
# ----------------------------

PLOT_CFG = {
    # Global styling / output
    "figsize": (8, 6),
    "title": "QBER/time - 10Hz, Night (no fibre)",
    "outfile": "C5_NightNoF_10Hz_EMvsNM.pdf",
    "main_xlim": (0, 200),
    "main_ylim": (0, 50),

    # Series labels (top plot)
    "top_labels": ["Emulation", "No Emulation"],

    # Series labels (bottom plot) - if you omit one series, only first label is used
    "bottom_labels": ["Emulation", "No Emulation"],

    # Whether to plot the second (orange) bottom series at all
    "plot_bottom_second_series": False,   # <- set False to omit DB_2/ts_2
    
    # Histogram label
    "hist_xlabel": "Prob. [%]",

    # Legend configuration (place legend on the upper histogram by default)
    # You can change this without touching plotting code.
    "legend": {
        "target": "hist_main",                 # "hist_main", "main", "sub", "hist_sub"
        "loc": "lower center",
        "bbox_to_anchor": (-1.64, 0.83),
        "fontsize": 12,
        "frameon": True,
        "ncol": 1
    },

    # Layout tweaks
    "gridspec": {
        "height_ratios": [3.5, 1],
        "width_ratios": [6.2, 1.4],
        "hspace": 0.015,
        "wspace": 0.015
    },

    # Histogram styling
    "hist_style": {
        "fill_alpha": 0.55,
        "sigma_band_alpha": 0.18,
        "lw": 0,
        "mean_lw": 2.8
    }
}

PLOT_CFG["hist_bins_top_scale"] = 1.25

# ----------------------------
# Histogram statistics glyph configuration
# ----------------------------

HIST_STYLE_CFG = {
    "fill_alpha": 0.43,
    "core_alpha": 0.57
}

STATS_TICK_CFG = {
    "x_offset": 0.201,     # + into gap/hist, - into main plot
    "tick_len": 0.013,     # figure fraction
    "mean_alpha": 1.0,
    "mean_lw": 2.5,

    "show_sigma": False,   # toggle
    "sigma_alpha": 0.2,
    "sigma_lw": 1.5,
    "sigma_ls": "-",
    "zorder": 30

}

# Bundle series inputs so the plotting cell stays generic
SERIES_TOP = [
    {"x": x1, "y": y1 * 100},
    {"x": x2, "y": y2 * 100},
]
SERIES_BOTTOM = [
    {"x": ts_1, "y": DB_1},
    {"x": ts_2, "y": DB_2},
]


In [23]:
# ----------------------------
# Load data
# ----------------------------

d1 = np.load('NNF_100_em_Numba.npy', allow_pickle=True).tolist()
d2 = np.load('NNF_100_nm_Numba.npy', allow_pickle=True).tolist()

x1 = d1[0][2][0]
y1 = d1[0][2][1]
x2 = d2[0][2][0]
y2 = d2[0][2][1]

DB_1 = Vis.convert_to_diff_db(np.load('rx_300m_night_2_long_etas.npy'))
ts_1 = [i / 1000 for i in range(len(DB_1))]

DB_2 = Vis.convert_to_diff_db(np.load('rx_300m_night_2_long_etas.npy'))
ts_2 = [i / 1000 for i in range(len(DB_2))]

# ----------------------------
# Plot configuration (edit THIS cell only)
# ----------------------------

PLOT_CFG = {
    # Global styling / output
    "figsize": (8, 6),
    "title": "QBER/time - 100Hz, Night (no fibre)",
    "outfile": "C5_NightNoF_100Hz_EMvsNM.pdf",
    "main_xlim": (0, 200),
    "main_ylim": (0, 50),

    # Series labels (top plot)
    "top_labels": ["Emulation", "No Emulation"],

    # Series labels (bottom plot) - if you omit one series, only first label is used
    "bottom_labels": ["Emulation", "No Emulation"],

    # Whether to plot the second (orange) bottom series at all
    "plot_bottom_second_series": False,   # <- set False to omit DB_2/ts_2
    
    # Histogram label
    "hist_xlabel": "Prob. [%]",

    # Legend configuration (place legend on the upper histogram by default)
    # You can change this without touching plotting code.
    "legend": {
        "target": "hist_main",                 # "hist_main", "main", "sub", "hist_sub"
        "loc": "lower center",
        "bbox_to_anchor": (-1.64, 0.83),
        "fontsize": 12,
        "frameon": True,
        "ncol": 1
    },

    # Layout tweaks
    "gridspec": {
        "height_ratios": [3.5, 1],
        "width_ratios": [6.2, 1.4],
        "hspace": 0.015,
        "wspace": 0.015
    },

    # Histogram styling
    "hist_style": {
        "fill_alpha": 0.55,
        "sigma_band_alpha": 0.18,
        "lw": 0,
        "mean_lw": 2.8
    }
}

PLOT_CFG["hist_bins_top_scale"] = 1.25
STATS_TICK_CFG["show_sigma"] = False
# ----------------------------
# Histogram statistics glyph configuration
# ----------------------------

HIST_STYLE_CFG = {
    "fill_alpha": 0.43,
    "core_alpha": 0.57
}

STATS_TICK_CFG = {
    "x_offset": 0.201,     # + into gap/hist, - into main plot
    "tick_len": 0.013,     # figure fraction
    "mean_alpha": 1.0,
    "mean_lw": 2.5,

    "show_sigma": False,   # toggle
    "sigma_alpha": 0.2,
    "sigma_lw": 1.5,
    "sigma_ls": "-",
    "zorder": 30

}
HIST_STYLE_CFG = {
    "outline_alpha": 0.70,
    "outline_lw": 2.5,
    "core_alpha": 0.65
}
# Bundle series inputs so the plotting cell stays generic
SERIES_TOP = [
    {"x": x1, "y": y1 * 100},
    {"x": x2, "y": y2 * 100},
]
SERIES_BOTTOM = [
    {"x": ts_1, "y": DB_1},
    {"x": ts_2, "y": DB_2},
]


In [773]:
# ----------------------------
# Matplotlib style
# ----------------------------
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=18)
plt.rc('axes', labelsize=16)

fig = plt.figure(figsize=PLOT_CFG["figsize"])

gs_cfg = PLOT_CFG["gridspec"]
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=gs_cfg["height_ratios"],
    width_ratios=gs_cfg["width_ratios"],
    hspace=gs_cfg["hspace"],
    wspace=gs_cfg["wspace"]
)

ax_main      = fig.add_subplot(gs[0, 0])
ax_sub       = fig.add_subplot(gs[1, 0], sharex=ax_main)
ax_hist_main = fig.add_subplot(gs[0, 1], sharey=ax_main)
ax_hist_sub  = fig.add_subplot(gs[1, 1], sharey=ax_sub)

# ----------------------------
# Main plots (top)
# ----------------------------
top_labels = PLOT_CFG["top_labels"]
line_top_0, = ax_main.plot(SERIES_TOP[0]["x"], SERIES_TOP[0]["y"], label=top_labels[0])
line_top_1, = ax_main.plot(SERIES_TOP[1]["x"], SERIES_TOP[1]["y"], label=top_labels[1])

ax_main.set_xlim(*PLOT_CFG["main_xlim"])
ax_main.set_ylim(*PLOT_CFG["main_ylim"])
ax_main.tick_params(axis='both', labelsize=14)

# Bottom plot (Δη)
bottom_labels = PLOT_CFG["bottom_labels"]
ax_sub.plot(SERIES_BOTTOM[0]["x"], SERIES_BOTTOM[0]["y"],
            color=line_top_0.get_color(), label=bottom_labels[0])

plot_bottom_second = PLOT_CFG["plot_bottom_second_series"]
if plot_bottom_second:
    ax_sub.plot(SERIES_BOTTOM[1]["x"], SERIES_BOTTOM[1]["y"],
                color=line_top_1.get_color(), label=bottom_labels[1])
ax_sub.tick_params(axis='both', labelsize=12)

ax_main.grid(True, axis='y', alpha=0.5, linewidth=0.8)
ax_sub.grid(True, axis='y', alpha=0.35, linewidth=0.8)
# ----------------------------
# Labels / title
# ----------------------------
ax_sub.set_xlabel('Time [$s$]')
ax_main.set_ylabel('QBER [$\\%$]')
ax_sub.set_ylabel('$\\Delta\\eta$ [db]')
ax_main.set_title(PLOT_CFG["title"])

plt.setp(ax_main.get_xticklabels(), visible=False)
ax_main.tick_params(axis='x', which='both', length=0)

# ----------------------------
# Histogram axes formatting
# ----------------------------
for axh in (ax_hist_main, ax_hist_sub):
    axh.grid(False)
    axh.tick_params(axis='y', labelleft=False, left=False)
    axh.tick_params(axis='y', which='both', length=0)
    axh.minorticks_off()
    axh.tick_params(axis='both', labelsize=9)  # de-emphasise histogram numbers
    for spine in axh.spines.values():
        spine.set_linewidth(0.45)

# Avoid tick/label collision between histogram panels
ax_hist_main.tick_params(axis='x', which='both',length=0,labelbottom=False)
ax_hist_main.set_xlabel("")

ax_hist_sub.set_xlabel(PLOT_CFG["hist_xlabel"])
ax_hist_main.grid(True, axis='y', alpha=0.5, linewidth=0.6)
ax_hist_sub.grid(True, axis='y', alpha=0.35, linewidth=0.6)
# ----------------------------
# Build bins and plot histograms
# ----------------------------
fig.canvas.draw()

ymin_m, ymax_m = ax_main.get_ylim()
ymin_s, ymax_s = ax_sub.get_ylim()

# Select bottom series to include for histogram (mirrors line plotting)
bottom_hist_data = [SERIES_BOTTOM[0]["y"]]
if plot_bottom_second:
    bottom_hist_data.append(SERIES_BOTTOM[1]["y"])

nb_main_base = choose_bins_freedman_diaconis(
    np.r_[SERIES_TOP[0]["y"], SERIES_TOP[1]["y"]], ymin_m, ymax_m
)

nb_main = int(nb_main_base * PLOT_CFG["hist_bins_top_scale"])

nb_sub = choose_bins_freedman_diaconis(
    np.concatenate([_finite(d) for d in bottom_hist_data]), ymin_s, ymax_s
)

bins_main = np.linspace(ymin_m, ymax_m, nb_main)
bins_sub  = np.linspace(ymin_s, ymax_s, nb_sub)

# Colors
c0 = line_top_0.get_color()
c1 = line_top_1.get_color()

# Histogram style: expects keys like fill_alpha/core_alpha/marker_size
# e.g. HIST_STYLE_CFG = {"fill_alpha":0.18, "core_alpha":0.65, "marker_size":16}
p1, mu1, s1 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[0]["y"], bins_main, c0, **HIST_STYLE_CFG)
p2, mu2, s2 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[1]["y"], bins_main, c1, **HIST_STYLE_CFG)

p3, mu3, s3 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[0]["y"], bins_sub, c0, **HIST_STYLE_CFG)
p4, mu4, s4 = (0.0, np.nan, np.nan)

if plot_bottom_second:
    p4, mu4, s4 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[1]["y"], bins_sub, c1, **HIST_STYLE_CFG)

# mean/σ ticks anchored to right edge of the main plot column
draw_stats_ticks_at_main_rhs(fig, ax_main, ax_hist_main, mu1, s1, c0, STATS_TICK_CFG)
draw_stats_ticks_at_main_rhs(fig, ax_main, ax_hist_main, mu2, s2, c1, STATS_TICK_CFG)

draw_stats_ticks_at_main_rhs(fig, ax_sub,  ax_hist_sub,  mu3, s3, c0, STATS_TICK_CFG)
if plot_bottom_second:
    draw_stats_ticks_at_main_rhs(fig, ax_sub, ax_hist_sub, mu4, s4, c1, STATS_TICK_CFG)


ax_main.set_axisbelow(True)
ax_sub.set_axisbelow(True)
# Shared histogram x-limits
peak = max(p1, p2, p3, p4)
xmax = 1.10 * peak if peak > 0 else 1.0
ax_hist_main.set_xlim(0, xmax)
ax_hist_sub.set_xlim(0, xmax)

# ----------------------------
# Histogram ticks: a few ticks above 0, hide label at 0
# ----------------------------
# Choose ticks in percent, then convert to fraction units
pct_ticks = [0, 5, 10, 15]  # adjust if you want
frac_ticks = [p / 100 for p in pct_ticks]

# Keep only ticks within [0, xmax]
frac_ticks = [t for t in frac_ticks if t <= xmax]

for axh in (ax_hist_main, ax_hist_sub):
    axh.set_xticks(frac_ticks)

# Hide the 0 label but keep the tick mark
def _hide_zero_percent(x, pos):
    if np.isclose(x, 0):
        return ""
    return f"{int(round(100 * x))}"

for axh in (ax_hist_main, ax_hist_sub):
    axh.xaxis.set_major_formatter(FuncFormatter(_hide_zero_percent))


# ----------------------------
# Tick padding: separate main vs histogram
# ----------------------------
ax_main.tick_params(axis='x', which='major', pad=4)
ax_sub.tick_params(axis='x', which='major', pad=4)
ax_hist_main.tick_params(axis='x', which='major', pad=0)
ax_hist_sub.tick_params(axis='x', which='major', pad=0)

# ----------------------------
# Legend (configurable target + position) with μ,σ for top series
# ----------------------------
legend_cfg = dict(PLOT_CFG.get("legend", {}))
target = legend_cfg.pop("target", "hist_main")

legend_targets = {
    "main": ax_main,
    "sub": ax_sub,
    "hist_main": ax_hist_main,
    "hist_sub": ax_hist_sub
}
ax_for_legend = legend_targets.get(target, ax_hist_main)

legend_labels = [
    f"{top_labels[0]}: μ={mu1:.2f}%, σ={s1:.2f}%",
    f"{top_labels[1]}: μ={mu2:.2f}%, σ={s2:.2f}%"
]
apply_legend(ax_for_legend, [line_top_0, line_top_1], legend_labels, legend_cfg)

# ----------------------------
# Margins / save
# ----------------------------
fig.subplots_adjust(left=0.12, right=0.95, bottom=0.10, top=0.92)

plt.show()
plt.savefig(PLOT_CFG["outfile"], dpi=300)



In [800]:
def h2(x):
    x = np.asarray(x, dtype=float)
    x = np.clip(x, 1e-12, 1 - 1e-12)
    return -(x*np.log2(x) + (1-x)*np.log2(1-x))

def secret_fraction_from_qber(E, f_ec=1.16, phase_error_equal_bit=True):
    """
    Visibility-limited TF-QKD proxy:
      r = 1 - f*h2(E) - h2(e_ph)
    with e_ph ~= E (simple, defensible for profiling).
    """
    E = np.asarray(E, dtype=float)
    e_ph = E if phase_error_equal_bit else E
    r = 1 - f_ec*h2(E) - h2(e_ph)
    return np.maximum(0.0, r)

def metrics_from_r_V_E(r, V, E, V0=0.90, E0_pct=10.0):
    r = np.asarray(r)
    V = np.asarray(V)
    E = np.asarray(E)

    m = np.isfinite(r) & np.isfinite(V) & np.isfinite(E)
    r = r[m]; V = V[m]; E = E[m]

    if r.size == 0:
        return dict(U=np.nan, rbar=np.nan, r5=np.nan, UV=np.nan, UE=np.nan)

    return {
        "U":   float(np.mean(r > 0)),
        "rbar": float(np.mean(r)),
        "r5":  float(np.percentile(r, 5)),
        "UV":  float(np.mean(V > V0)),
        "UE":  float(np.mean(E * 100 < E0_pct))  # QBER below threshold
    }



In [802]:
PLOT_CFG["f_ec"] = 1.16      # error-correction inefficiency
PLOT_CFG["V0"]   = 0.95      # visibility threshold for Pr[V>V0]

In [ ]:
# ----------------------------
# TF-QKD profiling metrics from top-series QBER/visibility
# ----------------------------
f_ec = PLOT_CFG.get("f_ec", 1.16)
V0   = PLOT_CFG.get("V0", 0.90)

def qber_percent_to_E(qber_percent):
    return np.asarray(qber_percent, dtype=float) / 100.0

def E_to_V(E):
    return 1.0 - 2.0*np.asarray(E, dtype=float)

# Build E(t), V(t), r(t) for each TOP series
E0 = SERIES_TOP[0]["y"] / 100.0
V0_t = 1 - 2 * E0
r0 = secret_fraction_from_qber(E0, f_ec=PLOT_CFG["f_ec"])

E1 = SERIES_TOP[1]["y"] / 100.0
V1_t = 1 - 2 * E1
r1 = secret_fraction_from_qber(E1, f_ec=PLOT_CFG["f_ec"])

m0 = metrics_from_r_V_E(r0, V0_t, E0, V0=PLOT_CFG["V0"], E0_pct=10.0)
m1 = metrics_from_r_V_E(r1, V1_t, E1, V0=PLOT_CFG["V0"], E0_pct=10.0)


# ----------------------------
# Metrics text boxes: top-right of upper histogram, stacked
# ----------------------------
def fmt_metrics(name, m, V0):
    return (
        f"{name}\n"
        f"U = {m['U']*100:5.1f}%\n"
        f"r̄ = {m['rbar']:.3f}\n"
        f"r₅ = {m['r5']:.3f}\n"
        f"Pr[QBER<10%] = {m['UE']*100:5.1f}%\n"
        f"Pr[V>{V0:.2f}] = {m['UV']*100:5.1f}%"
    )



txt0 = fmt_metrics(top_labels[0], m0, V0)
txt1 = fmt_metrics(top_labels[1], m1, V0)

# Place at top-right, stacked (axes coords). Adjust y positions if you want more spacing.
ax_hist_main.text(
    0.98, 0.98, txt0,
    transform=ax_hist_main.transAxes,
    ha="right", va="top",
    fontsize=9,
    color=c0,
    bbox=dict(facecolor="white", alpha=0.78, edgecolor="0.7", boxstyle="round,pad=0.25")
)

ax_hist_main.text(
    0.98, 0.62, txt1,   # second box below the first
    transform=ax_hist_main.transAxes,
    ha="right", va="top",
    fontsize=9,
    color=c1,
    bbox=dict(facecolor="white", alpha=0.78, edgecolor="0.7", boxstyle="round,pad=0.25")
)

In [27]:
# ----------------------------
# Matplotlib style
# ----------------------------
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=18)
plt.rc('axes', labelsize=16)

fig = plt.figure(figsize=PLOT_CFG["figsize"])

gs_cfg = PLOT_CFG["gridspec"]
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=gs_cfg["height_ratios"],
    width_ratios=gs_cfg["width_ratios"],
    hspace=gs_cfg["hspace"],
    wspace=gs_cfg["wspace"]
)

ax_main      = fig.add_subplot(gs[0, 0])
ax_sub       = fig.add_subplot(gs[1, 0], sharex=ax_main)
ax_hist_main = fig.add_subplot(gs[0, 1], sharey=ax_main)
ax_hist_sub  = fig.add_subplot(gs[1, 1], sharey=ax_sub)

# ----------------------------
# Main plots (top)
# ----------------------------
top_labels = PLOT_CFG["top_labels"]
line_top_0, = ax_main.plot(SERIES_TOP[0]["x"], SERIES_TOP[0]["y"], label=top_labels[0])
line_top_1, = ax_main.plot(SERIES_TOP[1]["x"], SERIES_TOP[1]["y"], label=top_labels[1])

ax_main.set_xlim(*PLOT_CFG["main_xlim"])
ax_main.set_ylim(*PLOT_CFG["main_ylim"])
ax_main.tick_params(axis='both', labelsize=14)

QBER_THRESH = 10.0  # percent

ax_main.axhline(
    QBER_THRESH,
    linestyle="--",
    linewidth=2.6,
    color="0.25",
    alpha=0.95,
    zorder=6
)
ax_main.axhspan(
    QBER_THRESH, ax_main.get_ylim()[1],
    color="0.35", alpha=0.25, zorder=0
)
# Bottom plot (Δη)
bottom_labels = PLOT_CFG["bottom_labels"]
ax_sub.plot(SERIES_BOTTOM[0]["x"], SERIES_BOTTOM[0]["y"],
            color=line_top_0.get_color(), label=bottom_labels[0])

plot_bottom_second = PLOT_CFG["plot_bottom_second_series"]
if plot_bottom_second:
    ax_sub.plot(SERIES_BOTTOM[1]["x"], SERIES_BOTTOM[1]["y"],
                color=line_top_1.get_color(), label=bottom_labels[1])
ax_sub.tick_params(axis='both', labelsize=12)

ax_main.grid(True, axis='y', alpha=0.5, linewidth=0.8)
ax_sub.grid(True, axis='y', alpha=0.35, linewidth=0.8)
# ----------------------------
# Labels / title
# ----------------------------
ax_sub.set_xlabel('Time [$s$]')
ax_main.set_ylabel('QBER [$\\%$]')
ax_sub.set_ylabel('$\\Delta\\eta$ [db]')
ax_main.set_title(PLOT_CFG["title"])

plt.setp(ax_main.get_xticklabels(), visible=False)
ax_main.tick_params(axis='x', which='both', length=0)

# ----------------------------
# Histogram axes formatting
# ----------------------------
for axh in (ax_hist_main, ax_hist_sub):
    axh.grid(False)
    axh.tick_params(axis='y', labelleft=False, left=False)
    axh.tick_params(axis='y', which='both', length=0)
    axh.minorticks_off()
    axh.tick_params(axis='both', labelsize=9)  # de-emphasise histogram numbers
    for spine in axh.spines.values():
        spine.set_linewidth(0.45)

# Avoid tick/label collision between histogram panels
ax_hist_main.tick_params(axis='x', which='both',length=0,labelbottom=False)
ax_hist_main.set_xlabel("")

ax_hist_sub.set_xlabel(PLOT_CFG["hist_xlabel"])
ax_hist_main.grid(True, axis='y', alpha=0.5, linewidth=0.6)
ax_hist_sub.grid(True, axis='y', alpha=0.35, linewidth=0.6)
QBER_THRESH = 10.0  # %
ax_hist_main.axhline(
    QBER_THRESH,
    linestyle="--",
    linewidth=1.4,
    color="0.25",
    alpha=0.95,
    zorder=10
)


# ----------------------------
# Build bins and plot histograms
# ----------------------------
fig.canvas.draw()

ymin_m, ymax_m = ax_main.get_ylim()
ymin_s, ymax_s = ax_sub.get_ylim()

# Select bottom series to include for histogram (mirrors line plotting)
bottom_hist_data = [SERIES_BOTTOM[0]["y"]]
if plot_bottom_second:
    bottom_hist_data.append(SERIES_BOTTOM[1]["y"])

nb_main_base = choose_bins_freedman_diaconis(
    np.r_[SERIES_TOP[0]["y"], SERIES_TOP[1]["y"]], ymin_m, ymax_m
)

nb_main = int(nb_main_base * PLOT_CFG["hist_bins_top_scale"])

nb_sub = choose_bins_freedman_diaconis(
    np.concatenate([_finite(d) for d in bottom_hist_data]), ymin_s, ymax_s
)

bins_main = np.linspace(ymin_m, ymax_m, nb_main)
bins_sub  = np.linspace(ymin_s, ymax_s, nb_sub)

# Colors
c0 = line_top_0.get_color()
c1 = line_top_1.get_color()

# Histogram style: expects keys like fill_alpha/core_alpha/marker_size
# e.g. HIST_STYLE_CFG = {"fill_alpha":0.18, "core_alpha":0.65, "marker_size":16}
p1, mu1, s1 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[0]["y"], bins_main, c0, **HIST_STYLE_CFG)
p2, mu2, s2 = fraction_hist_with_stats(ax_hist_main, SERIES_TOP[1]["y"], bins_main, c1, **HIST_STYLE_CFG)

p3, mu3, s3 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[0]["y"], bins_sub, c0, **HIST_STYLE_CFG)
p4, mu4, s4 = (0.0, np.nan, np.nan)

if plot_bottom_second:
    p4, mu4, s4 = fraction_hist_with_stats(ax_hist_sub, SERIES_BOTTOM[1]["y"], bins_sub, c1, **HIST_STYLE_CFG)

# mean/σ ticks anchored to right edge of the main plot column
draw_stats_ticks_at_main_rhs(fig, ax_main, ax_hist_main, mu1, s1, c0, STATS_TICK_CFG)
draw_stats_ticks_at_main_rhs(fig, ax_main, ax_hist_main, mu2, s2, c1, STATS_TICK_CFG)

draw_stats_ticks_at_main_rhs(fig, ax_sub,  ax_hist_sub,  mu3, s3, c0, STATS_TICK_CFG)
if plot_bottom_second:
    draw_stats_ticks_at_main_rhs(fig, ax_sub, ax_hist_sub, mu4, s4, c1, STATS_TICK_CFG)


ax_main.set_axisbelow(True)
ax_sub.set_axisbelow(True)
# Shared histogram x-limits
peak = max(p1, p2, p3, p4)
xmax = 1.10 * peak if peak > 0 else 1.0
ax_hist_main.set_xlim(0, xmax)
ax_hist_sub.set_xlim(0, xmax)

# ----------------------------
# Histogram ticks: a few ticks above 0, hide label at 0
# ----------------------------
# Choose ticks in percent, then convert to fraction units
pct_ticks = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]  # adjust if you want
frac_ticks = [p / 100 for p in pct_ticks]

# Keep only ticks within [0, xmax]
frac_ticks = [t for t in frac_ticks if t <= xmax]

for axh in (ax_hist_main, ax_hist_sub):
    axh.set_xticks(frac_ticks)

# Hide the 0 label but keep the tick mark
def _hide_zero_percent(x, pos):
    if np.isclose(x, 0):
        return ""
    return f"{int(round(100 * x))}"

for axh in (ax_hist_main, ax_hist_sub):
    axh.xaxis.set_major_formatter(FuncFormatter(_hide_zero_percent))

ax_hist_main.axhspan(
    QBER_THRESH, ax_hist_main.get_ylim()[1],
    color="0.35", alpha=0.2, zorder=0
)
# ----------------------------
# Tick padding: separate main vs histogram
# ----------------------------
ax_main.tick_params(axis='x', which='major', pad=4)
ax_sub.tick_params(axis='x', which='major', pad=4)
ax_hist_main.tick_params(axis='x', which='major', pad=0)
ax_hist_sub.tick_params(axis='x', which='major', pad=0)

# ----------------------------
# Legend (configurable target + position) with μ,σ for top series
# ----------------------------
legend_cfg = dict(PLOT_CFG.get("legend", {}))
target = legend_cfg.pop("target", "hist_main")

legend_targets = {
    "main": ax_main,
    "sub": ax_sub,
    "hist_main": ax_hist_main,
    "hist_sub": ax_hist_sub
}
ax_for_legend = legend_targets.get(target, ax_hist_main)

legend_labels = [
    f"{top_labels[0]}: μ={mu1:.2f}%, σ={s1:.2f}%",
    f"{top_labels[1]}: μ={mu2:.2f}%, σ={s2:.2f}%"
]
apply_legend(ax_for_legend, [line_top_0, line_top_1], legend_labels, legend_cfg)


# Margins / save
# ----------------------------
fig.subplots_adjust(left=0.12, right=0.95, bottom=0.10, top=0.92)

plt.show()
#plt.savefig(PLOT_CFG["outfile"], dpi=300)



In [943]:
d1[1]


VisibilityTauResult(taus=array([3.00e-04, 4.00e-04, 5.00e-04, 6.00e-04, 7.00e-04, 8.00e-04,
       9.00e-04, 1.00e-03, 2.00e-03, 3.00e-03, 4.00e-03, 5.00e-03,
       6.00e-03, 7.00e-03, 8.00e-03, 9.00e-03, 1.00e-02, 1.10e-02,
       1.20e-02, 1.30e-02, 1.40e-02, 1.50e-02, 1.60e-02, 1.70e-02,
       1.90e-02, 2.00e-02, 2.20e-02, 2.30e-02, 2.50e-02, 2.70e-02,
       2.80e-02, 3.10e-02, 3.30e-02, 3.50e-02, 3.80e-02, 4.00e-02,
       4.30e-02, 4.60e-02, 5.00e-02, 5.30e-02, 5.70e-02, 6.10e-02,
       6.60e-02, 7.10e-02, 7.60e-02, 8.10e-02, 8.70e-02, 9.30e-02,
       1.00e-01, 1.07e-01, 1.15e-01, 1.23e-01, 1.32e-01, 1.42e-01,
       1.52e-01, 1.63e-01, 1.75e-01, 1.87e-01, 2.01e-01, 2.15e-01,
       2.31e-01, 2.48e-01, 2.66e-01, 2.85e-01, 3.05e-01, 3.27e-01,
       3.51e-01, 3.76e-01, 4.04e-01, 4.33e-01, 4.64e-01, 4.98e-01,
       5.34e-01, 5.72e-01, 6.14e-01, 6.58e-01, 7.05e-01, 7.56e-01,
       8.11e-01, 8.70e-01, 9.33e-01, 1.00e+00]), n_windows=array([1999998,  500000,  400000,  333333,  2

In [962]:
import numpy as np
import matplotlib.pyplot as plt

def plot_visibility_vs_tau_with_poisson_regions(
    res,
    *,
    title="Visibility vs Tau (Poisson-dominated regions highlighted)",
    N_crit: float = 50.0,          # counts/window below which shot noise is inherently large
    poisson_frac: float = 0.6,     # if >=60% of variance explained by Poisson -> call it Poisson-dominated
    shade_alpha: float = 0.18,
    show_sigma_panel: bool = True,
):
    """
    Takes VisibilityTauResult `res` (from lockin_visibility_summary_vs_tau_mle_numba)
    and highlights tau regions where the large 10–90% band is likely dominated by Poisson noise.

    Poisson test is based on:
      - expected counts per window N = res.expected_counts_per_window
      - empirical sigma from (V_p90 - V_p10)/2.563
      - predicted Poisson sigma ~ sqrt((1 - V^2)/N)

    A tau is flagged Poisson-dominated if:
      (N < N_crit) OR (sigma_pois^2 / sigma_emp^2 >= poisson_frac)
    """

    taus = np.asarray(res.taus, float)
    Vmean = np.asarray(res.V_mean, float)
    Vmed  = np.asarray(res.V_median, float)
    Vmax  = np.asarray(res.V_max, float)
    Vmin  = np.asarray(res.V_min, float)
    Vp10  = np.asarray(res.V_p10, float)
    Vp90  = np.asarray(res.V_p90, float)
    Nexp  = np.asarray(res.expected_counts_per_window, float)

    # finite mask
    m = np.isfinite(taus) & np.isfinite(Vmean) & np.isfinite(Vp10) & np.isfinite(Vp90) & np.isfinite(Nexp) & (taus > 0)
    if not np.any(m):
        raise ValueError("No finite tau/visibility data to plot.")

    taus = taus[m]
    Vmean = Vmean[m]
    Vmed  = Vmed[m]
    Vmax  = Vmax[m]
    Vmin  = Vmin[m]
    Vp10  = Vp10[m]
    Vp90  = Vp90[m]
    Nexp  = Nexp[m]

    # ---- Empirical sigma from 10–90 band
    band = Vp90 - Vp10
    sigma_emp = band / 2.5631031310892007  # 2*(z_0.90)=2*1.28155...

    # ---- Poisson-predicted sigma (rule-of-thumb)
    # Clamp V into [0,1] for the formula, because it is a physical variance model
    Vc = np.clip(Vmean, 0.0, 1.0)
    Nsafe = np.maximum(Nexp, 1.0)
    sigma_pois = np.sqrt(np.maximum(0.0, 1.0 - Vc*Vc) / Nsafe)

    # ---- How much of variance could be explained by Poisson?
    var_emp = np.maximum(sigma_emp*sigma_emp, 1e-20)
    var_pois = sigma_pois*sigma_pois
    frac_explained = np.clip(var_pois / var_emp, 0.0, 1.0)

    poisson_dom = (Nexp < N_crit) | (frac_explained >= poisson_frac)

    # helper: shade contiguous regions where poisson_dom True
    def _shade_regions(ax, x, mask, *, alpha=0.18):
        # x is sorted in log axis order already (taus increasing)
        idx = np.where(mask)[0]
        if idx.size == 0:
            return
        # find contiguous runs
        runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
        for r in runs:
            a = x[r[0]]
            b = x[r[-1]]
            # extend to next point edge for nicer shading
            ax.axvspan(a, b, alpha=alpha)

    # ---- Plot: either one panel or two panels
    if show_sigma_panel:
        fig, ax = plt.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
        fig.set_size_inches(8, 6)
    else:
        fig, ax = plt.subplots(1, 1)
        fig.set_size_inches(8, 6)
        ax = [ax]

    # ===== Top: visibility summary =====
    ax0 = ax[0]
    ax0.plot(taus, Vmean, label="Vmean", linewidth=1.6)
    ax0.plot(taus, Vmed,  label="Vmedian", linewidth=1.2)
    ax0.plot(taus, Vmax,  label="Vmax", linewidth=1.0)
    ax0.plot(taus, Vmin,  label="Vmin", linewidth=1.0)
    ax0.fill_between(taus, Vp10, Vp90, alpha=0.20, label="10–90% band")

    _shade_regions(ax0, taus, poisson_dom, alpha=shade_alpha)

    ax0.set_xscale("log")
    ax0.set_ylim(-0.05, 1.25)
    ax0.set_ylabel("Visibility")
    ax0.set_title(title)
    ax0.grid(True, alpha=0.25)
    ax0.legend(loc="best")

    # remove excess x padding
    ax0.set_xlim(taus[0], taus[-1])

    # ===== Bottom: noise diagnostics =====
    if show_sigma_panel:
        ax1 = ax[1]
        ax1.plot(taus, sigma_emp, label=r"$\sigma_{\mathrm{emp}}$ from 10–90 band", linewidth=1.6)
        ax1.plot(taus, sigma_pois, label=r"$\sigma_{\mathrm{Pois}} \approx \sqrt{(1-V^2)/N}$", linewidth=1.3)
        ax1.plot(taus, frac_explained, label=r"Poisson variance fraction", linewidth=1.2)

        _shade_regions(ax1, taus, poisson_dom, alpha=shade_alpha)

        ax1.set_xscale("log")
        ax1.set_yscale("log")
        ax1.set_xlabel("Tau (s)")
        ax1.set_ylabel("σ / fraction")
        ax1.grid(True, alpha=0.25)
        ax1.legend(loc="best")

    fig.set_constrained_layout(True)

    # Print a quick summary
    print("[Poisson-region diagnostics]")
    print(f"  N_crit = {N_crit} counts/window")
    print(f"  poisson_frac threshold = {poisson_frac:.2f}")
    print(f"  Fraction of taus flagged Poisson-dominated: {np.mean(poisson_dom):.2%}")
    print(f"  Smallest tau flagged: {taus[poisson_dom][0] if np.any(poisson_dom) else np.nan}")
    print(f"  Largest tau flagged:  {taus[poisson_dom][-1] if np.any(poisson_dom) else np.nan}")

    plt.show()

    # return arrays in case you want to annotate/report
    return {
        "taus": taus,
        "sigma_emp": sigma_emp,
        "sigma_pois": sigma_pois,
        "poisson_frac_explained": frac_explained,
        "poisson_dominated_mask": poisson_dom,
        "Nexp": Nexp,
    }


In [948]:
diag = plot_visibility_vs_tau_with_poisson_regions(
    d1[1],
    N_crit=50,           # tune for your regime
    poisson_frac=0.6,    # tune for strictness
    show_sigma_panel=True
)

[Poisson-region diagnostics]
  N_crit = 50 counts/window
  poisson_frac threshold = 0.60
  Fraction of taus flagged Poisson-dominated: 9.76%
  Smallest tau flagged: 0.00030000000000000003
  Largest tau flagged:  0.001


In [950]:
import numpy as np
import matplotlib.pyplot as plt

def plot_visibility_vs_tau_with_poisson_and_mle_error(
    res,
    *,
    title="Visibility vs Tau (Poisson-dominated regions + MLE error estimate)",
    N_crit: float = 150.0,         # good default for your regime (see notes)
    poisson_frac: float = 0.6,
    shade_alpha: float = 0.18,
    mle_k: float = 1.0,            # scale factor for the MLE sigma model (keep 1.0 unless calibrating)
    show_sigma_panel: bool = True,
):
    """
    Plots V(tau) summaries and highlights Poisson-dominated regions.
    Adds an estimated MLE-only sigma_V(tau) ~ sqrt((1 - V^2)/N).

    Uses:
      - res.V_p10/res.V_p90 for empirical spread
      - res.expected_counts_per_window for expected photons per window
    """

    taus = np.asarray(res.taus, float)
    Vmean = np.asarray(res.V_mean, float)
    Vmed  = np.asarray(res.V_median, float)
    Vmax  = np.asarray(res.V_max, float)
    Vmin  = np.asarray(res.V_min, float)
    Vp10  = np.asarray(res.V_p10, float)
    Vp90  = np.asarray(res.V_p90, float)
    Nexp  = np.asarray(res.expected_counts_per_window, float)

    m = np.isfinite(taus) & np.isfinite(Vmean) & np.isfinite(Vp10) & np.isfinite(Vp90) & np.isfinite(Nexp) & (taus > 0)
    if not np.any(m):
        raise ValueError("No finite tau/visibility data to plot.")

    taus = taus[m]
    Vmean = Vmean[m]
    Vmed  = Vmed[m]
    Vmax  = Vmax[m]
    Vmin  = Vmin[m]
    Vp10  = Vp10[m]
    Vp90  = Vp90[m]
    Nexp  = Nexp[m]

    # ---- empirical sigma from 10–90 band
    band = Vp90 - Vp10
    sigma_emp = band / 2.5631031310892007  # 2*(z_0.90)=2*1.28155...

    # ---- Poisson-pred sigma (rule-of-thumb)
    Vc = np.clip(Vmean, 0.0, 1.0)
    Nsafe = np.maximum(Nexp, 1.0)
    sigma_pois = np.sqrt(np.maximum(0.0, 1.0 - Vc*Vc) / Nsafe)

    # ---- MLE sigma estimate (same scaling, optionally with a tunable factor)
    # In practice, MLE and Poisson scaling are very similar; this just makes it explicit as "MLE error".
    sigma_mle = mle_k * sigma_pois

    # ---- fraction of empirical variance explained by Poisson/MLE
    var_emp = np.maximum(sigma_emp*sigma_emp, 1e-20)
    frac_explained = np.clip((sigma_pois*sigma_pois) / var_emp, 0.0, 1.0)

    poisson_dom = (Nexp < N_crit) | (frac_explained >= poisson_frac)

    def _shade_regions(ax, x, mask, *, alpha=0.18):
        idx = np.where(mask)[0]
        if idx.size == 0:
            return
        runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
        for r in runs:
            ax.axvspan(x[r[0]], x[r[-1]], alpha=alpha)

    # ---- figure layout
    if show_sigma_panel:
        fig, ax = plt.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [2, 1]})
        fig.set_size_inches(8, 6)
        ax0, ax1 = ax
    else:
        fig, ax0 = plt.subplots(1, 1)
        fig.set_size_inches(8, 6)
        ax1 = None

    # ===== Top: visibility summaries =====
    ax0.plot(taus, Vmean, label="Vmean", linewidth=1.7)
    ax0.plot(taus, Vmed,  label="Vmedian", linewidth=1.2)
    ax0.plot(taus, Vmax,  label="Vmax", linewidth=1.0)
    ax0.plot(taus, Vmin,  label="Vmin", linewidth=1.0)
    ax0.fill_between(taus, Vp10, Vp90, alpha=0.20, label="10–90% band")

    # ---- MLE error band around Vmean (±1σ)
    V_lo = Vmean - sigma_mle
    V_hi = Vmean + sigma_mle
    ax0.fill_between(taus, V_lo, V_hi, alpha=0.15, label="Estimated MLE ±1σ")

    # ---- shade Poisson-dominated regions
    _shade_regions(ax0, taus, poisson_dom, alpha=shade_alpha)

    ax0.set_xscale("log")
    ax0.set_ylim(-0.05, 1.25)
    ax0.set_ylabel("Visibility")
    ax0.set_title(title)
    ax0.grid(True, alpha=0.25)
    ax0.legend(loc="best")
    ax0.set_xlim(taus[0], taus[-1])

    # ===== Bottom: sigma diagnostics =====
    if ax1 is not None:
        ax1.plot(taus, sigma_emp, label=r"$\sigma_{\mathrm{emp}}$ (from 10–90 band)", linewidth=1.6)
        ax1.plot(taus, sigma_pois, label=r"$\sigma_{\mathrm{Pois}}$", linewidth=1.3)
        ax1.plot(taus, sigma_mle, label=r"$\sigma_{\mathrm{MLE, est}}$", linewidth=1.2, linestyle="--")
        ax1.plot(taus, frac_explained, label="Poisson variance fraction", linewidth=1.1)

        _shade_regions(ax1, taus, poisson_dom, alpha=shade_alpha)

        ax1.set_xscale("log")
        ax1.set_yscale("log")
        ax1.set_xlabel("Tau (s)")
        ax1.set_ylabel("σ / fraction")
        ax1.grid(True, alpha=0.25)
        ax1.legend(loc="best")

    fig.set_constrained_layout(True)
    plt.show()

    print("[Noise diagnostics]")
    print(f"  Mean rate implied by expected_counts/tau ~ {np.nanmedian(Nexp/taus):.2e} counts/s")
    print(f"  N_crit = {N_crit:.1f} counts/window")
    print(f"  poisson_frac threshold = {poisson_frac:.2f}")
    print(f"  Fraction of taus flagged Poisson-dominated: {np.mean(poisson_dom):.2%}")

    return {
        "taus": taus,
        "sigma_emp": sigma_emp,
        "sigma_pois": sigma_pois,
        "sigma_mle": sigma_mle,
        "poisson_frac_explained": frac_explained,
        "poisson_dominated_mask": poisson_dom,
        "Nexp": Nexp,
    }


In [1014]:
diag = plot_visibility_vs_tau_with_poisson_and_mle_error(
    d2[1],
    N_crit=150,
    poisson_frac=0.8,
    mle_k=1.0,
    show_sigma_panel=True
)

[Noise diagnostics]
  Mean rate implied by expected_counts/tau ~ 2.96e+04 counts/s
  N_crit = 150.0 counts/window
  poisson_frac threshold = 0.80
  Fraction of taus flagged Poisson-dominated: 14.63%
